# **Instalação**

In [ ]:
!pip install -q git+https://github.com/bdcdo/raspe.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


# **Raspagem de dados**

In [ ]:
import logging
logging.getLogger("FOLHA").setLevel(logging.INFO)

import raspe

folha = raspe.folha()

dados = folha.raspar(
    pesquisa=["bolsonaro", "lula", "eleições", "eleitoral"],
    site="todos",
    data_inicio="2022-10-01",
    data_fim="2022-11-30",
)

print(f"Total bruto: {len(dados)}")

# remover duplicatas (notícias que mencionam os dois nomes)
termo_agrupado = dados.groupby("link")["termo_busca"].apply(lambda x: ", ".join(sorted(set(x))))
dados = dados.drop_duplicates(subset="link").reset_index(drop=True)
dados = dados.merge(termo_agrupado.rename("termos_busca"), on="link")
dados = dados.drop(columns=["termo_busca"])

print(f"Total após remover duplicatas: {len(dados)}")
dados.head()

Streaming output truncated to the last 5000 lines.
2026-09-24 20:07:44,494 - FOLHA - DEBUG - {'q': 'eleições', 'site': 'todos', 'periodo': 'personalizado', 'sr': 1351, 'sd': '01/10/2022', 'ed': '30/11/2022'}
2026-09-24 20:07:44,494 - FOLHA - DEBUG - {'q': 'eleições', 'site': 'todos', 'periodo': 'personalizado', 'sr': 1351, 'sd': '01/10/2022', 'ed': '30/11/2022'}
2026-09-24 20:07:44,494 - FOLHA - DEBUG - {'q': 'eleições', 'site': 'todos', 'periodo': 'personalizado', 'sr': 1351, 'sd': '01/10/2022', 'ed': '30/11/2022'}
2026-09-24 20:07:44,494 - FOLHA - DEBUG - {'q': 'eleições', 'site': 'todos', 'periodo': 'personalizado', 'sr': 1351, 'sd': '01/10/2022', 'ed': '30/11/2022'}
2026-09-24 20:07:45,103 - FOLHA - DEBUG - Response status: 200
2026-09-24 20:07:45,103 - FOLHA - DEBUG - Response status: 200
2026-09-24 20:07:45,103 - FOLHA - DEBUG - Response status: 200
2026-09-24 20:07:45,103 - FOLHA - DEBUG - Response status: 200
2026-09-24 20:07:45,103 - FOLHA - DEBUG - Response status: 200
2026-0

Total bruto: 16204
Total após remover duplicatas: 7058


,link,titulo,resumo,data,termos_busca
0,https://www1.folha.uol.com.br/blogs/claudio-he...,Trump e Bolsonaro vão morar na Itália?,...,9.nov.2022 às 19h56,bolsonaro
1,https://www1.folha.uol.com.br/ambiente/2022/11...,"Desejada por Lula, parceria com Indonésia e Re...",divulgada em veículos da imprensa internaciona...,9.nov.2022 às 19h55,"bolsonaro, eleitoral, lula"
2,https://www1.folha.uol.com.br/poder/2022/11/lu...,Lula diz que cabe a Bolsonaro reconhecer derro...,"O ministro presidiu as eleições gerais, encerr...",9.nov.2022 às 19h54,"bolsonaro, eleitoral, eleições, lula"
3,https://www1.folha.uol.com.br/poder/2022/11/lu...,Lula desarma golpismo bolsonarista sem embate ...,"eletrônicas, que cumpriu o papel de deixar sus...",9.nov.2022 às 19h50,"bolsonaro, eleitoral, eleições, lula"
4,https://www1.folha.uol.com.br/poder/2022/11/de...,Relatório da Defesa não aponta fraude em eleiç...,relatório sobre a fiscalização do processo ele...,9.nov.2022 às 19h47,"bolsonaro, eleitoral, eleições, lula"


# **Filtrar apenas Opinião**

In [ ]:
dados_opiniao = dados[dados["link"].str.contains("/opiniao/", na=False)].reset_index(drop=True)
print(f"Notícias de Opinião: {len(dados_opiniao)}")
dados_opiniao.head()

Notícias de Opinião: 202


,link,titulo,resumo,data,termos_busca
0,https://www1.folha.uol.com.br/opiniao/2022/11/...,"Desta vez, vamos garantir que os EUA respeitem...",Moro tornou-se ministro da Justiça quatro dias...,9.nov.2022 às 15h38,"bolsonaro, eleições, lula"
1,https://www1.folha.uol.com.br/opiniao/2022/11/...,Desordem do dia,A nota ainda condena as autoridades que cercei...,13.nov.2022 às 21h30,"bolsonaro, lula"
2,https://www1.folha.uol.com.br/opiniao/2022/11/...,Chance aproveitada,"A seu favor, será facílimo desenvolver polític...",18.nov.2022 às 21h30,"bolsonaro, lula"
3,https://www1.folha.uol.com.br/opiniao/2022/11/...,A PEC da Transição é necessária para ajustar o...,"Só no governo Bolsonaro, foi furado em R$ 795 ...",18.nov.2022 às 21h00,"bolsonaro, lula"
4,https://www1.folha.uol.com.br/opiniao/2022/11/...,A PEC da Transição é necessária para ajustar o...,"Com mudanças anuais nas regras, o governo Jair...",18.nov.2022 às 21h00,"bolsonaro, lula"


# **Diagnóstico do HTML antes de rodar tudo**

In [ ]:
import requests
from bs4 import BeautifulSoup
import json

url_teste = dados_opiniao["link"].iloc[0]
print("URL testada:", url_teste)

headers = {"User-Agent": "Mozilla/5.0"}
r = requests.get(url_teste, headers=headers, timeout=10)
soup = BeautifulSoup(r.text, "html.parser")

print("\n--- Meta tags relevantes ---")
for m in soup.find_all("meta"):
    nome = m.get("name") or m.get("property")
    if nome and ("author" in nome.lower() or "byline" in nome.lower() or "writer" in nome.lower()):
        print(nome, "->", m.get("content"))

print("\n--- JSON-LD encontrado ---")
for script in soup.find_all("script", type="application/ld+json"):
    try:
        data = json.loads(script.string)
        print(json.dumps(data, indent=2, ensure_ascii=False)[:1000])
        print("---")
    except (json.JSONDecodeError, TypeError):
        continue

print("\n--- Elementos com 'autor'/'author' na classe ---")
for tag in soup.find_all(class_=True):
    classes = " ".join(tag.get("class", []))
    if "autor" in classes.lower() or "author" in classes.lower():
        print(tag.name, classes, "->", tag.get_text(strip=True)[:100])

print("\n--- Links para /colunistas/ ou /colunas/ ---")
for a in soup.find_all("a", href=True):
    if "/colunistas/" in a["href"] or "/colunas/" in a["href"]:
        print(a["href"], "->", a.get_text(strip=True))

URL testada: https://www1.folha.uol.com.br/opiniao/2022/11/desta-vez-vamos-garantir-que-os-eua-respeitem-a-democracia-no-brasil.shtml

--- Meta tags relevantes ---

--- JSON-LD encontrado ---
{
  "@context": "http://schema.org",
  "@type": [
    "CreativeWork",
    "OpinionNewsArticle"
  ],
  "url": "https://www1.folha.uol.com.br/opiniao/2022/11/desta-vez-vamos-garantir-que-os-eua-respeitem-a-democracia-no-brasil.shtml",
  "mainEntityOfPage": "https://www1.folha.uol.com.br/opiniao/2022/11/desta-vez-vamos-garantir-que-os-eua-respeitem-a-democracia-no-brasil.shtml",
  "headline": "Desta vez, vamos garantir que os EUA respeitem a democracia no Brasil",
  "description": "Devemos ficar atentos com o que o governo americano fará nos próximos meses e anos",
  "datePublished": "2022-11-09T15:38:00Z",
  "image": {
    "@type": "ImageObject",
    "url": "https://f.i.uol.com.br/fotografia/2022/10/31/1667186705635f40113a31f_1667186705_3x2_md.jpg",
    "width": "768",
    "height": "512"
  },
  "au

# **Coleta de autor e texto**




In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time

GENERICOS = {"folha.uol.com.br", "folha de s.paulo", "folha de sao paulo", "uol", "folhapress"}

def pegar_autor_e_texto(url, timeout=15):
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        r = requests.get(url, headers=headers, timeout=timeout)
        r.encoding = "utf-8"  # corrige o problema de acentuação
        soup = BeautifulSoup(r.text, "html.parser")

        # --- autor ---
        autor = None
        candidatos = []
        for script in soup.find_all("script", type="application/ld+json"):
            try:
                data = json.loads(script.string)
                if isinstance(data, dict) and "author" in data:
                    a = data["author"]
                    if isinstance(a, dict) and a.get("name"):
                        candidatos.append(a["name"].strip())
                    elif isinstance(a, list):
                        for item in a:
                            if isinstance(item, dict) and item.get("name"):
                                candidatos.append(item["name"].strip())
            except (json.JSONDecodeError, TypeError):
                continue

        meta = soup.find("meta", attrs={"name": "author"})
        if meta and meta.get("content"):
            candidatos.append(meta["content"].strip())

        for c in candidatos:
            if c.lower() not in GENERICOS:
                autor = c
                break

        # --- texto ---
        texto = None
        corpo = soup.find("div", class_="c-news__body")
        if corpo:
            texto = corpo.get_text(" ", strip=True)

        return autor, texto

    except requests.RequestException:
        return None, None


autores = []
textos = []
total = len(dados_opiniao)

for i, link in enumerate(dados_opiniao["link"], start=1):
    autor, texto = pegar_autor_e_texto(link)
    autores.append(autor)
    textos.append(texto)
    if i % 10 == 0 or i == total:
        print(f"{i}/{total} processadas")
    time.sleep(1)

dados_opiniao["autor"] = autores
dados_opiniao["texto"] = textos
dados_opiniao.head()

10/202 processadas
20/202 processadas
30/202 processadas
40/202 processadas
50/202 processadas
60/202 processadas
70/202 processadas
80/202 processadas
90/202 processadas
100/202 processadas
110/202 processadas
120/202 processadas
130/202 processadas
140/202 processadas
150/202 processadas
160/202 processadas
170/202 processadas
180/202 processadas
190/202 processadas
200/202 processadas
202/202 processadas


,link,titulo,resumo,data,termos_busca,autor,texto
0,https://www1.folha.uol.com.br/opiniao/2022/11/...,"Desta vez, vamos garantir que os EUA respeitem...",Moro tornou-se ministro da Justiça quatro dias...,9.nov.2022 às 15h38,"bolsonaro, eleições, lula",Mark Weisbrot,"O povo brasileiro, com seu voto, tirou um mons..."
1,https://www1.folha.uol.com.br/opiniao/2022/11/...,Desordem do dia,A nota ainda condena as autoridades que cercei...,13.nov.2022 às 21h30,"bolsonaro, lula",None,"Por quase duas semanas, as Forças Armadas soub..."
2,https://www1.folha.uol.com.br/opiniao/2022/11/...,Chance aproveitada,"A seu favor, será facílimo desenvolver polític...",18.nov.2022 às 21h30,"bolsonaro, lula",None,"Recebido com grande expectativa positiva, o pr..."
3,https://www1.folha.uol.com.br/opiniao/2022/11/...,A PEC da Transição é necessária para ajustar o...,"Só no governo Bolsonaro, foi furado em R$ 795 ...",18.nov.2022 às 21h00,"bolsonaro, lula",Débora Freire,A reconstrução do país passa pela correção de ...
4,https://www1.folha.uol.com.br/opiniao/2022/11/...,A PEC da Transição é necessária para ajustar o...,"Com mudanças anuais nas regras, o governo Jair...",18.nov.2022 às 21h00,"bolsonaro, lula",Magno Karl e Mano Ferreira,O governo Lula 3 quer começar com um furo no t...


# **Remover linhas sem autor (editoriais) e checar texto**

In [ ]:
antes = len(dados_opiniao)
dados_opiniao = dados_opiniao[dados_opiniao["autor"].notna()].reset_index(drop=True)
depois = len(dados_opiniao)
print(f"Removidos {antes - depois} editoriais sem autor. Restaram {depois} notícias assinadas.")
dados_opiniao.head()

Removidos 94 editoriais sem autor. Restaram 108 notícias assinadas.


,link,titulo,resumo,data,termos_busca,autor,texto
0,https://www1.folha.uol.com.br/opiniao/2022/11/...,"Desta vez, vamos garantir que os EUA respeitem...",Moro tornou-se ministro da Justiça quatro dias...,9.nov.2022 às 15h38,"bolsonaro, eleições, lula",Mark Weisbrot,"O povo brasileiro, com seu voto, tirou um mons..."
1,https://www1.folha.uol.com.br/opiniao/2022/11/...,A PEC da Transição é necessária para ajustar o...,"Só no governo Bolsonaro, foi furado em R$ 795 ...",18.nov.2022 às 21h00,"bolsonaro, lula",Débora Freire,A reconstrução do país passa pela correção de ...
2,https://www1.folha.uol.com.br/opiniao/2022/11/...,A PEC da Transição é necessária para ajustar o...,"Com mudanças anuais nas regras, o governo Jair...",18.nov.2022 às 21h00,"bolsonaro, lula",Magno Karl e Mano Ferreira,O governo Lula 3 quer começar com um furo no t...
3,https://www1.folha.uol.com.br/opiniao/2022/10/...,O risco da desqualificação dos institutos de p...,"Já Bolsonaro teve 43,20%, cinco pontos acima d...",3.out.2022 às 15h59,"bolsonaro, eleições, lula",Samuel Mendonça,Finalizado o primeiro turno das eleições presi...
4,https://www1.folha.uol.com.br/opiniao/2022/10/...,A fé e o governo Bolsonaro,Cito aqui alguns arranjos religiosos no govern...,26.out.2022 às 21h00,"bolsonaro, eleições, lula",Ronaldo de Almeida,Religião na política é tema recorrente há um b...


# **Salvar**

In [ ]:
dados_opiniao.to_excel("folha_opiniao.xlsx", index=False)
from google.colab import files
files.download("folha_opiniao.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>